# Agent 3: Food Safety Report Writer

**Purpose**: Translate machine-readable batch risk summary into human-readable investigation report for:
- Food safety officers
- Procurement managers
- Regulators

**Input**: 
- `batch_risk_summary.json` (from Agent 2)
- `enriched_samples.json` (from Agent 1)

**Output**: 
- `food_safety_report.json` (structured)
- `food_safety_report.md` (human-readable markdown)

## Step 1: Setup & Imports

In [ ]:
import os
import sys
import json
import time
from typing import Any, Dict, List, Optional
from datetime import datetime
import logging

# LangGraph imports
from langgraph.graph import StateGraph, START, END

# Add foodguard lib to path
sys.path.insert(0, '..')
import foodguard_lib as fgl
from foodguard_lib import (
    FOOD_SAFETY_REPORT_JSON, FOOD_SAFETY_REPORT_MD, BASE_URL_MODEL, OPENAI_API_KEY, LLM
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✓ Imports successful")

import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Configure vLLM environment
os.environ["BASE_URL"] = BASE_URL_MODEL
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Initialize LLM client for report generation
try:
    llm = ChatOpenAI(
        model=LLM,
        base_url=BASE_URL_MODEL,
        api_key=OPENAI_API_KEY,
        temperature=0.7,
        max_tokens=2048,
    )
    print(f"✓ vLLM ({LLM}) configured for batch analysis")
except Exception as e:
    print(f"⚠ Warning: Could not connect to vLLM server: {e}")
    llm = None

## Step 2: Report Template Functions

In [ ]:
def generate_executive_summary(
    batch_id: str,
    total_samples: int,
    batch_decision: str,
    adulterant_dist: Dict[str, float]
) -> str:
    """
    Generate 1-2 paragraph executive summary.
    """
    summary = f"""
## Executive Summary

**Batch ID**: {batch_id}
**Sample Count**: {total_samples}
**Analysis Date**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
**Batch Decision**: **{batch_decision}**

FoodGuard AI has completed automated analysis of {total_samples} milk samples from batch {batch_id}
using multi-modal sensor data (vision, e-nose, e-tongue) cross-referenced against NABL-certified reference materials.

"""

    # Add summary line
    main_adulterant = max(adulterant_dist.items(), key=lambda x: x[1])
    if main_adulterant[0] == 'authentic':
        summary += f"All {total_samples} samples were certified **AUTHENTIC**. Batch approved for distribution.\n"
    else:
        summary += f"{main_adulterant[1]:.0%} of samples show **{main_adulterant[0].upper()}**. " \
                   f"Batch flagged for further action per decision.\n"

    return summary

print("✓ Executive summary function defined")

In [ ]:
# ========== LLM-Enhanced Report Narrative Generation ==========

def use_llm_for_narrative_generation(batch_summary: Dict, technical_data: Dict) -> str:
    """
    Use vLLM to generate clear, regulatory-compliant narrative for report.
    Translates technical data into human-readable findings.
    """
    if llm is None:
        return "Report generation requires vLLM connection."

    prompt = f"""
You are a regulatory compliance officer writing food safety investigation reports.

BATCH SUMMARY:
{json.dumps(batch_summary, indent=2)}

TECHNICAL FINDINGS:
{json.dumps(technical_data, indent=2)}

TASK:
Generate a 2-3 paragraph technical narrative that:
1. Explains findings in regulatory language
2. States clear risk level and implications
3. Recommends next steps
4. Maintains professional tone suitable for regulators

Write only the narrative paragraphs (no markdown headings):
"""

    try:
        response = llm.invoke([HumanMessage(content=prompt)])
        return response.content
    except Exception as e:
        print(f"LLM narrative generation failed: {e}")
        return "Narrative generation pending LLM availability."

print("✓ LLM report generation functions defined")

## Step 3: Detailed Findings Section

In [ ]:
def generate_detailed_findings(
    enriched_samples: List[Dict[str, Any]],
    batch_patterns: Dict[str, Any]
) -> str:
    """
    Generate detailed findings table with sample-level data.
    """
    findings = "\n## Detailed Sample Analysis\n\n"
    findings += "| Sample ID | Ground Truth | Risk Level | Confidence | Status | Note |\n"
    findings += "|-----------|--------------|------------|------------|--------|------|\n"

    for sample in enriched_samples:
        sample_id = sample['sample_id']
        ground_truth = sample['ground_truth']
        risk = sample['risk_level']
        conf = sample['confidence']
        status = sample['status']
        note = sample.get('note', '—')[:50]  # Truncate for table

        findings += f"| {sample_id} | {ground_truth} | {risk} | {conf:.0%} | {status} | {note} |\n"

    # Add statistics
    patterns = batch_patterns
    findings += f"\n### Batch Statistics\n"
    findings += f"- **Average Confidence**: {patterns['confidence_avg']:.0%}\n"
    findings += f"- **Confidence Range**: {patterns['confidence_range'][0]:.0%} — {patterns['confidence_range'][1]:.0%}\n"
    findings += f"- **Inconclusive Samples**: {patterns['inconclusive_count']}/{patterns['total_samples']} ({patterns['inconclusive_percentage']:.0%})\n"
    findings += f"- **Low Confidence Samples**: {patterns['low_confidence_count']}/{patterns['total_samples']}\n"

    return findings

print("✓ Detailed findings function defined")

## Step 4: Decision & Recommendations Section

In [ ]:
def generate_decision_section(
    batch_decision: str,
    decision_reasoning: List[str],
    recommended_actions: List[str]
) -> str:
    """
    Generate decision and action section.
    """
    decision_section = f"\n## Decision\n\n"

    # Decision with color coding (for markdown rendering)
    decision_colors = {
        'ACCEPT': '✅ APPROVED',
        'CONDITIONAL_ACCEPT': '⚠️ CONDITIONAL APPROVAL',
        'QUARANTINE': '🚨 QUARANTINE',
        'REJECT': '❌ REJECT',
        'RETEST': '🔄 RETEST REQUIRED',
    }

    decision_emoji = decision_colors.get(batch_decision, batch_decision)
    decision_section += f"**Batch Status**: {decision_emoji}\n\n"

    # Reasoning
    decision_section += f"### Decision Rationale\n\n"
    if decision_reasoning:
        for reason in decision_reasoning:
            decision_section += f"- {reason}\n"
    else:
        decision_section += f"- No specific issues identified.\n"

    # Recommendations
    decision_section += f"\n### Recommended Actions\n\n"
    if recommended_actions:
        for i, action in enumerate(recommended_actions, 1):
            decision_section += f"{i}. {action}\n"
    else:
        decision_section += f"1. No specific actions required.\n"

    return decision_section

print("✓ Decision section function defined")

## Step 5: Retest Section

In [ ]:
def generate_retest_section(retest_analysis: Dict[str, Any]) -> str:
    """
    Generate retest recommendations section.
    """
    if not retest_analysis['retest_required']:
        return "\n## Retest Analysis\n\nNo retest required.\n"

    retest_section = f"\n## Retest Analysis\n\n"
    retest_section += f"**Samples Flagged**: {retest_analysis['samples_flagged']}\n\n"
    retest_section += "| Sample ID | Current Prediction | Confidence | Retest Reasons |\n"
    retest_section += "|-----------|-------------------|------------|-----------------|\n"

    for sample in retest_analysis['samples']:
        sample_id = sample['sample_id']
        pred = sample['current_prediction']
        conf = sample['confidence']
        reasons = ', '.join(sample['retest_reasons'])

        retest_section += f"| {sample_id} | {pred} | {conf:.0%} | {reasons} |\n"

    retest_section += f"\n**Recommendation**: {retest_analysis['samples'][0]['recommendation'] if retest_analysis['samples'] else 'N/A'}\n"

    return retest_section

print("✓ Retest section function defined")

## Step 6: Methodology Section

In [ ]:
def generate_methodology_section() -> str:
    """
    Generate technical methodology section for auditors.
    """
    methodology = f"""
## Methodology

### Analysis Pipeline

1. **Agent 1: CRM Correlator**
   - Raw sensor data scored against NABL Certified Reference Materials (CRMs)
   - Cross-modal feature matching (vision, e-nose, e-tongue)
   - Differentiators applied for ambiguous cases (DIFF-1 through DIFF-5)
   - Per-sample confidence calculated with modality-based adjustments

2. **Agent 2: Batch Risk Assessor**
   - Adulterant distribution across batch
   - Pattern detection (e.g., % with critical contamination)
   - Batch-level risk determination
   - Retest flags for inconclusive/low-confidence samples

3. **Agent 3: Food Safety Report Writer**
   - Human-readable translation of results
   - Executive summary for regulatory submission
   - Detailed findings with sample-level audit trail

### Reference Standards
- **Issuing Body**: NABL Accredited Dairy Lab, India
- **CRM Version**: 1.0
- **Confirmatory Methods**: Cryoscopy, Gerber fat test, Kjeldahl protein, Lactometer SNF

### Confidence Adjustments
- Single modality: confidence capped at 65%
- Two modalities: confidence capped at 80%
- Three modalities: no cap (max 95%)
- Differentiator used: -5% penalty
- Top-2 candidates within 10%: -5% penalty
"""
    return methodology

print("✓ Methodology section function defined")

## Step 7: Generate Full Markdown Report

In [ ]:
def generate_markdown_report(
    batch_id: str,
    enriched_samples: List[Dict[str, Any]],
    batch_risk_summary: Dict[str, Any]
) -> str:
    """
    Generate complete markdown report.
    """
    patterns = batch_risk_summary['batch_patterns']
    decision_data = batch_risk_summary

    report = f"""# FoodGuard AI — Milk Safety Investigation Report

**Report Generated**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
**Report ID**: {batch_id}
**System**: FoodGuard Automated Food Safety Platform

---

"""

    # Add sections
    report += generate_executive_summary(
        batch_id,
        len(enriched_samples),
        decision_data['batch_decision'],
        patterns['adulterant_distribution']
    )

    report += generate_detailed_findings(enriched_samples, patterns)

    report += generate_decision_section(
        decision_data['batch_decision'],
        decision_data['decision_reasoning'],
        decision_data['recommended_actions']
    )

    report += generate_retest_section(decision_data['retest_analysis'])

    report += generate_methodology_section()

    # Footer
    report += f"""
---

## Disclaimer

This report is generated by automated AI analysis and should be reviewed by a qualified food safety officer
before regulatory submission. Confirmatory testing with reference methods is recommended for critical findings.

**For questions, contact**: FoodGuard AI Support
**Platform**: Version 1.0 (LangGraph Multi-Agent)
"""

    return report

print("✓ Markdown report generation defined")

## Step 8: LangGraph Node Function

In [ ]:
def report_writer_node(state: Dict[str, Any]) -> Dict[str, Any]:
    """
    Agent 3 node: Food Safety Report Writer.
    Generates human-readable investigation report from batch analysis.
    """
    start_time = time.time()

    enriched_samples = state.get('enriched_samples', [])
    batch_risk_summary = state.get('batch_risk_summary', {})
    batch_id = state.get('batch_id', 'unknown')

    logger.info(f"Report Writer generating report for batch {batch_id}")

    # Generate markdown report
    markdown_report = generate_markdown_report(batch_id, enriched_samples, batch_risk_summary)
    logger.info(f"Markdown report generated ({len(markdown_report)} chars)")

    # Execution time
    execution_time = (time.time() - start_time) * 1000  # ms
    exec_id = fgl.generate_id('EXEC')

    # Build structured report
    food_safety_report = {
        'batch_id': batch_id,
        'report_id': fgl.generate_id('RPT'),
        'timestamp': datetime.now().isoformat(),
        'batch_decision': batch_risk_summary.get('batch_decision', 'UNKNOWN'),
        'total_samples': len(enriched_samples),
        'adulterant_distribution': batch_risk_summary['batch_patterns'].get('adulterant_distribution', {}),
        'average_confidence': batch_risk_summary['batch_patterns'].get('confidence_avg', 0.0),
        'decision_reasoning': batch_risk_summary.get('decision_reasoning', []),
        'recommended_actions': batch_risk_summary.get('recommended_actions', []),
        'sample_details': enriched_samples,
    }

    # Return updated state
    state['food_safety_report'] = food_safety_report
    state['markdown_report'] = markdown_report
    state['agent_execution_id'] = exec_id
    state['execution_time_ms'] = execution_time
    state['reasoning'] = f"Generated food safety report with {len(enriched_samples)} sample details and batch decision: {food_safety_report['batch_decision']}"

    logger.info(f"Report Writer complete")

    return state

print("✓ LangGraph node function defined")

## Step 9: Build LangGraph

In [ ]:
# Build graph
builder = StateGraph(dict)
builder.add_node("report_writer", report_writer_node)
builder.set_entry_point("report_writer")
builder.add_edge("report_writer", END)

graph = builder.compile()

print("✓ LangGraph compiled")
print("\nGraph structure:")
print(graph.get_graph().draw_ascii())

## Step 10: Test with Sample Data

In [ ]:
# Create test data simulating Agent 1 & 2 output
test_enriched_samples = [
    {
        'sample_id': 'S001',
        'ground_truth': 'authentic',
        'confidence': 0.92,
        'risk_level': 'None',
        'status': 'Safe',
        'note': 'All features within authentic CRM ranges',
    },
    {
        'sample_id': 'S002',
        'ground_truth': 'urea_added',
        'confidence': 0.85,
        'risk_level': 'High',
        'status': 'Unsafe',
        'note': 'High ammonia signal matched urea CRM',
    },
]

test_batch_risk_summary = {
    'batch_id': fgl.generate_batch_id(),
    'batch_decision': 'QUARANTINE',
    'decision_reasoning': ['50% of batch shows urea contamination'],
    'recommended_actions': ['Quarantine batch pending confirmatory testing', 'Contact supplier'],
    'batch_patterns': {
        'adulterant_distribution': {'authentic': 0.5, 'urea_added': 0.5},
        'confidence_avg': 0.88,
        'confidence_range': (0.85, 0.92),
        'inconclusive_count': 0,
        'low_confidence_count': 0,
        'total_samples': 2,
    },
    'retest_analysis': {
        'retest_required': False,
        'samples_flagged': 0,
        'samples': [],
    },
}

# Run graph
initial_state = {
    'enriched_samples': test_enriched_samples,
    'batch_risk_summary': test_batch_risk_summary,
    'batch_id': test_batch_risk_summary['batch_id'],
}

output_state = graph.invoke(initial_state)

print(f"✓ Graph executed successfully")
print(f"Report ID: {output_state['food_safety_report']['report_id']}")

## Step 11: Display Report Preview

In [ ]:
# Display markdown report
print("\n" + "="*80)
print("FOOD SAFETY REPORT (MARKDOWN PREVIEW)")
print("="*80)
print(output_state['markdown_report'])

## Step 12: Save Report to Files

In [ ]:
# Save structured JSON report
json_output_path = FOOD_SAFETY_REPORT_JSON
os.makedirs(os.path.dirname(json_output_path), exist_ok=True)

with open(json_output_path, 'w') as f:
    json.dump(output_state['food_safety_report'], f, indent=2)

print(f"✓ JSON report saved to: {json_output_path}")
print(f"  File size: {os.path.getsize(json_output_path) / 1024:.1f} KB")

# Save markdown report
md_output_path = FOOD_SAFETY_REPORT_MD

with open(md_output_path, 'w') as f:
    f.write(output_state['markdown_report'])

print(f"✓ Markdown report saved to: {md_output_path}")
print(f"  File size: {os.path.getsize(md_output_path) / 1024:.1f} KB")